# MEA Model - CMU-MOSI Dataset

Multimodal fusion approach for learning modality-Exclusive and modality-Agnostic representations

In [1]:
!git clone https://github.com/ranbeer06052009/DRCFNet-Research

Cloning into 'test'...
remote: Enumerating objects: 87, done.
remote: Counting objects: 100% (87/87), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 87 (delta 15), reused 75 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (87/87), 12.61 MiB | 30.52 MiB/s, done.
Resolving deltas: 100% (15/15), done.


In [3]:
import gdown

file_id = "1szKIqO0t3Be_W91xvf6aYmsVVUa7wDHU"
destination = "mosi_raw.pkl"

gdown.download(
    f"https://drive.google.com/uc?id={file_id}", destination, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1szKIqO0t3Be_W91xvf6aYmsVVUa7wDHU
From (redirected): https://drive.google.com/uc?id=1szKIqO0t3Be_W91xvf6aYmsVVUa7wDHU&confirm=t&uuid=cfc330a2-caf0-4986-8980-e0d2a06f95d9
To: /content/mosi_raw.pkl
100%|██████████| 357M/357M [00:03<00:00, 90.6MB/s]


'mosi_raw.pkl'

In [2]:
import sys
import torch
import matplotlib.pyplot as plt

sys.path.append('/content/DRCFNet-Research/src')

from loader import get_dataloader
from models.mea import MEA
from training.train_mea import mea_criterion, train_mea_loop, test_mea
from evaluation.performance import eval_affect

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Using device:", device)

Using device: cuda


In [4]:
# Load MOSI Data
train_data, valid_data, test_data = get_dataloader(
    '/content/mosi_raw.pkl',
    max_pad=True,
    max_seq_len=50
)

In [5]:
# Initialize MEA Model with Paper Hyperparameters for MOSI
model = MEA(
    dim_l=300,   # Text dimension
    dim_v=35,    # Vision dimension
    dim_a=74,    # Audio dimension
    d=40,        # Hidden Dimension d
    dh=64,       # Output Dimension dh
    n_heads=8,   # Attention Head
    mu=0.25      # Coefficient mu
).to(device)

# Note: The paper defines learning rate 1e-3, batch size 32, epochs 60, alpha 2e-2, beta 3e-2
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [ ]:
# Train MEA Model with Paper Hyperparameters
import torch.nn as nn

# As per the paper: epochs=60, alpha=2e-2, beta=3e-2
EPOCHS = 60
ALPHA = 2e-2
BETA = 3e-2

task_criterion = nn.L1Loss()

print(f"Starting Training for {EPOCHS} Epochs...")
model, history = train_mea_loop(
    model=model,
    train_loader=train_data,
    valid_loader=valid_data,
    task_criterion=task_criterion,
    optimizer=optimizer,
    epochs=EPOCHS,
    device=device,
    alpha=ALPHA,
    beta=BETA
)


In [7]:
# Test the Trained MEA Model
print("Evaluating best model on Test Set...")
import numpy as np
from sklearn.metrics import f1_score

metrics, preds, labels = test_mea(
    model=model,
    dataloader=test_data,
    task_criterion=torch.nn.L1Loss(),
    device=device,
    alpha=2e-2, # Trade-off Parameter alpha
    beta=3e-2,  # Trade-off Parameter beta
    return_preds=True
)

preds_np = preds.view(-1).cpu().numpy()
labels_np = labels.view(-1).cpu().numpy()

# ===== MAE =====
mae = np.mean(np.abs(preds_np - labels_np))

# ===== Corr =====
corr = np.corrcoef(preds_np, labels_np)[0][1]
corr = 0 if np.isnan(corr) else corr

# ===== Binary =====
pred_bin = (preds_np > 0).astype(int)
label_bin = (labels_np > 0).astype(int)

mask = labels_np != 0
acc2 = (pred_bin[mask] == label_bin[mask]).mean()
f1 = f1_score(label_bin[mask], pred_bin[mask], average='weighted')

print("\nFinal Evaluation (Random Weights!):")
print(f"MAE: {mae:.4f}")
print(f"Corr: {corr:.4f}")
print(f"Acc-2: {acc2*100:.2f}%")
print(f"F1: {f1*100:.2f}%")



Final Evaluation:
MAE: 1.0535
Corr: 0.5656
Acc-2: 72.10%
F1: 72.22%


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history["train_loss"], label="Train Loss", color="royalblue", linewidth=2)
plt.plot(history["val_loss"], label="Valid Loss", color="crimson", linewidth=2)
plt.title("Training vs Validation Loss", fontsize=12, fontweight='bold')
plt.xlabel("Epochs", fontsize=10)
plt.ylabel("Loss", fontsize=10)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history["train_task"], label="Train Task Loss (MAE)", color="royalblue", linewidth=2)
plt.title("Training Task Loss (MAE)", fontsize=12, fontweight='bold')
plt.xlabel("Epochs", fontsize=10)
plt.ylabel("Mean Absolute Error (MAE)", fontsize=10)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()

plt.tight_layout()
plt.savefig("train_val_curves_mea.jpg", dpi=300)
plt.show()

In [ ]:
# Generate and Save Advanced Interpretability Heatmaps for MEA
import torch
import torch.nn.functional as F
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Set model to evaluation mode
model.eval()

# Get a batch of test data
batch = next(iter(test_data))
vision, audio, text, labels = batch
vision = vision.to(device)
audio = audio.to(device)
text = text.to(device)

with torch.no_grad():
    logits, features = model(vision, audio, text, kg_features=None)

# -------------------------------------------------------------
# HEATMAP: Modality Decoupling Similarity Matrix (Exclusive vs Agnostic)
# -------------------------------------------------------------
h_e = features['h_e'] # [L, V, A]
h_a = features['h_a'] # [L, V, A]

# Average over batch to get the mean representation vectors (size dh = 64)
t_exc = h_e[0].mean(dim=0)
v_exc = h_e[1].mean(dim=0)
a_exc = h_e[2].mean(dim=0)

t_agn = h_a[0].mean(dim=0)
v_agn = h_a[1].mean(dim=0)
a_agn = h_a[2].mean(dim=0)

vectors = torch.stack([v_exc, v_agn, a_exc, a_agn, t_exc, t_agn], dim=0) # (6, 64)
# Normalize to compute cosine similarity
vectors_norm = F.normalize(vectors, p=2, dim=1)
sim_matrix = torch.mm(vectors_norm, vectors_norm.t()).cpu().numpy()

labels_heatmap = ['Visual-Exclusive', 'Visual-Agnostic', 'Acoustic-Exclusive', 'Acoustic-Agnostic', 'Text-Exclusive', 'Text-Agnostic']

plt.figure(figsize=(10, 8))
sns.heatmap(sim_matrix, xticklabels=labels_heatmap, yticklabels=labels_heatmap, annot=True, fmt=".2f", 
            cmap="coolwarm", vmin=-1.0, vmax=1.0, cbar_kws={'label': 'Cosine Similarity'})
plt.title("MEA Modality Decoupling Similarity Matrix (Exclusive vs Agnostic)", fontsize=11, fontweight='bold', pad=10)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig("mea_decoupling_heatmap.jpg", dpi=300)
plt.show()

print("Heatmap successfully saved as mea_decoupling_heatmap.jpg")

In [ ]:
# Save the model
torch.save(model.state_dict(), "mea_mosi_best.pt")
print("Model saved as mea_mosi_best.pt")